# Laboratorio L14 - Bias, scorciatoie e uso responsabile

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari: `L14_selezione_stage.csv`, `pinguini.csv`.

Il dataset `L14_selezione_stage.csv` è **simulato**: descrive 600 candidature fittizie a uno stage estivo selettivo e le decisioni prese negli anni precedenti da una commissione. Le colonne:

- `media_voti`, `assenze_perc` (percentuale di ore di assenza), `attivita_extra` (1 = ha svolto attività extrascolastiche)
- `quartiere`: centro o periferia
- `trasporto_scolastico`: 1 = usa il trasporto scolastico
- `colloquio_svolto`: 1 = ha sostenuto il colloquio finale
- `ammesso`: decisione storica della commissione (l'etichetta)
- `qualificato`: se il candidato soddisfaceva i requisiti dichiarati del bando (media, assenze, attività). Questa colonna esiste solo perché i dati sono simulati: nei dati reali l'informazione "vera" sul merito di solito non è disponibile

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

dati = pd.read_csv("L14_selezione_stage.csv")
dati.head()

## 1. Esplorare i dati storici (esercizio 1)

Confrontare, per quartiere, la frazione di candidati qualificati e la frazione di candidati ammessi.

- `(serie == "si").mean()`: frazione dei "si"
- `groupby("quartiere")[colonna].apply(funzione)`: applica la funzione a ogni gruppo

In [ ]:
frazione_si = lambda s: (s == "si").mean()
dati.groupby("quartiere")[["qualificato", "ammesso"]].agg(frazione_si).round(3)

Completare: tra i soli candidati **qualificati**, calcolare la frazione di ammessi per quartiere.

In [ ]:
qualificati = dati[dati["qualificato"] == ___]
tassi_storici = qualificati.groupby("quartiere")["ammesso"].agg(frazione_si).round(3)
print(tassi_storici)
controlla(tassi_storici["centro"] > tassi_storici["periferia"])

**Domanda**: che cosa mostrano i numeri sulle decisioni della commissione?

Risposta:

## 2. Addestrare un modello sulle decisioni storiche (esercizio 2)

Caratteristiche: media, assenze, attività, quartiere (codificato con `get_dummies`). Etichetta: `ammesso`.

In [ ]:
X = pd.get_dummies(dati[["media_voti", "assenze_perc", "attivita_extra", "quartiere"]], drop_first=True, dtype=int)
y = dati["ammesso"]
X_add, X_ver, y_add, y_ver, d_add, d_ver = train_test_split(X, y, dati, test_size=0.3, random_state=0, stratify=y)

modello = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_add, y_add)
print("accuratezza complessiva sulla verifica:", round(modello.score(X_ver, y_ver), 3))
print(pd.Series(modello.feature_importances_, index=X.columns).round(3))

`train_test_split` riceve qui anche la tabella completa `dati` e la divide nello stesso modo (`d_add`, `d_ver`): serve per sapere quartiere e qualifica dei candidati di verifica.

## 3. Valutare per gruppo (esercizio 3)

L'accuratezza complessiva è buona. Ma il modello tratta allo stesso modo i candidati qualificati dei due quartieri?

In [ ]:
d_ver = d_ver.copy()
d_ver["previsto"] = modello.predict(X_ver)
qual_ver = d_ver[d_ver["qualificato"] == "si"]
tassi_modello = qual_ver.groupby("quartiere")["previsto"].agg(frazione_si).round(3)
print("ammessi dal modello tra i qualificati:")
print(tassi_modello)

**Domande**

- Il modello ha "imparato" il comportamento della commissione? Da che cosa si vede?
- Un'accuratezza di 0,86 significa che il modello è giusto? Rispetto a che cosa è misurata l'accuratezza?

Risposte:

## 4. Togliere il quartiere basta? (esercizio 4)

Si addestra un modello senza la colonna `quartiere`, ma con `trasporto_scolastico`.

In [ ]:
X2 = dati[["media_voti", "assenze_perc", "attivita_extra", "trasporto_scolastico"]]
modello2 = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X2.loc[X_add.index], y_add)
d_ver["previsto2"] = modello2.predict(X2.loc[X_ver.index])
print(d_ver[d_ver["qualificato"] == "si"].groupby("quartiere")["previsto2"].agg(frazione_si).round(3))
print(pd.crosstab(dati["quartiere"], dati["trasporto_scolastico"], normalize="index").round(2))

**Domanda**: perché la differenza tra i quartieri non sparisce? Quali altre colonne possono "rappresentare" il quartiere? (Suggerimento: confrontare le assenze medie nei due quartieri con `dati.groupby("quartiere")["assenze_perc"].mean()`.)

Risposta:

## 5. Fuga di informazione (esercizio 5)

Si aggiunge la colonna `colloquio_svolto`.

In [ ]:
X3 = dati[["media_voti", "assenze_perc", "attivita_extra", "colloquio_svolto"]]
modello3 = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X3.loc[X_add.index], y_add)
acc3 = modello3.score(X3.loc[X_ver.index], ___)
print("accuratezza con colloquio_svolto:", round(acc3, 3))
controlla(acc3 == 1.0)
print(pd.crosstab(dati["colloquio_svolto"], dati["ammesso"]))

**Domanda**: perché l'accuratezza è perfetta? Il modello sarebbe utile per decidere su nuove candidature? Quando si conosce `colloquio_svolto`?

Risposta:

## 6. Scheda del modello dei pinguini (esercizio 6)

Completare nella cella Markdown seguente una scheda essenziale del classificatore dei pinguini costruito nel modulo 3.

| voce | contenuto |
|---|---|
| scopo del modello | |
| dati di addestramento (fonte, periodo, luogo, dimensione, licenza) | |
| caratteristiche usate | |
| modello e iperparametri | |
| prestazioni (su quali dati, con quale metrica) | |
| prestazioni per gruppo (per specie, per sesso) | |
| limiti noti e usi sconsigliati | |